# Evaluate CVE-only target_steps LR 2e-5 checkpoints

Evaluation-only kernel. Uses the checkpoints from:

`minhiw/vuln2action-t5-stage1-lr2e5-3ep`

Evaluates epoch 1, 2, 3 on 1,000 validation rows each.


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os
hits=sorted(glob.glob("/kaggle/input/**/flow_train.jsonl", recursive=True))
print("flow_train hits:", hits)
if not hits: raise FileNotFoundError("flow_train.jsonl not found")
FLOW_DIR=Path(hits[0]).parent
os.environ["FLOW_DIR"]=str(FLOW_DIR)
os.environ["TOKENIZERS_PARALLELISM"]="false"
print("FLOW_DIR:", FLOW_DIR)


In [ ]:
from pathlib import Path
import glob
hits=sorted(glob.glob("/kaggle/input/**/checkpoint-*", recursive=True))
ckpts=[]
for h in hits:
    p=Path(h)
    if p.is_dir() and p.name.startswith("checkpoint-") and "t5_stage1_target_steps_lr2e5_3ep" in str(p):
        ckpts.append(p)
# dedupe and sort by step
ckpts=sorted({str(p):p for p in ckpts}.values(), key=lambda p:int(p.name.split("-")[-1]))
print("checkpoints:")
for p in ckpts: print(" ", p)
if not ckpts: raise FileNotFoundError("no training checkpoints found")


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
results=[]
for ckpt in ckpts:
    out=Path(f"/kaggle/working/eval_{ckpt.name}")
    cmd=[
     sys.executable, "scripts/evaluate_t5.py",
     "--checkpoint", str(ckpt),
     "--dataset", os.environ["FLOW_DIR"],
     "--split", "validation",
     "--target-field", "target_steps",
     "--output-dir", str(out),
     "--max-input-length", "256",
     "--max-target-length", "384",
     "--batch-size", "4",
     "--num-beams", "4",
     "--limit", "1000",
     "--fp16",
    ]
    print("\n=== EVAL", ckpt.name, "===")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True, env=os.environ.copy())
    m=json.loads((out/"metrics.json").read_text(encoding="utf-8"))
    results.append((ckpt.name, m))
print("\n=== SUMMARY ===")
for name,m in results:
    print(name, "BLEU", m["bleu"], "ROUGE-L", m["rougeL_f1"], "corpus", m["bleu_corpus"])


In [ ]:
import json
from pathlib import Path
for out in sorted(Path("/kaggle/working").glob("eval_checkpoint-*")):
    m=json.loads((out/"metrics.json").read_text(encoding="utf-8"))
    rows=[json.loads(l) for l in (out/"predictions.jsonl").open(encoding="utf-8")]
    exact=sum(r["prediction"].strip()==r["reference"].strip() for r in rows)
    print(out.name, json.dumps({k:m[k] for k in ["bleu","bleu_corpus","rouge1_f1","rouge2_f1","rougeL_f1"]}, ensure_ascii=False), "exact", exact, "/", len(rows))
